# XGBoost Classifier — Practical Notebook

## Cheat-sheet hyperparameters
The supplied cheat sheet lists this XGBoost grid:

- `n_estimators`: `[50, 100, 200]`
- `learning_rate`: `[0.01, 0.1, 0.2]`
- `max_depth`: `[3, 5, 7]`
- `gamma`: `[0, 0.1, 0.3]`
- `subsample`: `[0.8, 1.0]`
- `colsample_bytree`: `[0.8, 1.0]`

That is **324 parameter combinations**. The cheat sheet's syntax also mentions `eval_metric="logloss"` and `random_state`. This notebook uses `eval_metric="logloss"` and `random_state=42`. `use_label_encoder` is deliberately omitted because it is deprecated/removed in modern XGBoost versions.

**Important:** XGBoost is an optional package. If it is not installed, run `%pip install xgboost` in a separate notebook cell and restart the kernel if prompted. This notebook is syntax-checked but not executed on your dataset.

## 1. Imports and configuration

In [ ]:

import os, glob, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate, learning_curve
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, cohen_kappa_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, average_precision_score,
    log_loss, RocCurveDisplay, PrecisionRecallDisplay
)
from scipy.stats import ttest_rel, wilcoxon

try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError(
        "XGBoost is not installed. Run `%pip install xgboost` in a separate cell, "
        "restart the kernel if needed, then rerun this notebook."
    ) from exc

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("XGBoost version:", __import__("xgboost").__version__)


## 2. Load CSV and select target

In [ ]:

csv_files = sorted(glob.glob("*.csv"))
if not csv_files:
    file_path = input("Enter the path to your CSV file: ").strip().strip('"')
elif len(csv_files) == 1:
    file_path = csv_files[0]
    print("Using CSV:", file_path)
else:
    print("CSV files found:")
    for i, name in enumerate(csv_files):
        print(f"{i}: {name}")
    file_path = csv_files[int(input("Enter the number of the CSV to use: ").strip())]

df = pd.read_csv(file_path)
print("Shape:", df.shape)
display(df.head())
print("Columns:", df.columns.tolist())

target_col = input("Enter the exact target column name: ").strip()
if target_col not in df.columns:
    raise ValueError(f"Target column {target_col!r} was not found.")

before = len(df)
df = df.drop_duplicates().dropna(subset=[target_col]).copy()
print(f"Removed {before - len(df)} duplicate/missing-target rows.")

X = df.drop(columns=[target_col]).copy()
y_raw = df[target_col].astype(str)
if X.shape[1] == 0:
    raise ValueError("No feature columns remain after separating the target.")

label_encoder = LabelEncoder()
y = pd.Series(label_encoder.fit_transform(y_raw), index=df.index, name=target_col)
class_names = [str(v) for v in label_encoder.classes_]
n_classes = len(class_names)
print("Classes:", class_names)
if n_classes < 2:
    raise ValueError("Classification requires at least two target classes.")


## 3. Exploratory data analysis

In [ ]:

print("Data types:")
display(df.dtypes.to_frame("dtype"))
print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("\nNumeric summary:")
display(df.describe(include=[np.number]).T)

ax = y.value_counts().sort_index().plot(kind="bar", figsize=(8, 4), title="Target class distribution")
ax.set_xlabel("Encoded class")
ax.set_ylabel("Count")
ax.set_xticks(range(n_classes))
ax.set_xticklabels(class_names, rotation=45, ha="right")
plt.tight_layout()
plt.show()

numeric_eda = X.select_dtypes(include=np.number).columns.tolist()
if numeric_eda:
    X[numeric_eda[:12]].hist(figsize=(12, 8), bins=20)
    plt.suptitle("Numeric feature distributions (up to 12 features)")
    plt.tight_layout()
    plt.show()

categorical_eda = X.select_dtypes(exclude=np.number).columns.tolist()
for col in categorical_eda[:5]:
    X[col].astype(str).value_counts().head(15).plot(
        kind="bar", figsize=(8, 3), title=f"Top categories: {col}"
    )
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()


## 4. Split data and preprocess
Tree-based XGBoost does not require feature scaling. Numeric features use median imputation; categorical features use most-frequent imputation and one-hot encoding. The preprocessing is inside the pipeline so it is fitted separately within each CV fold.

In [ ]:

if y.value_counts().min() < 2:
    raise ValueError("Each class needs at least 2 rows for stratified splitting and cross-validation.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=True)

transformers = []
if numeric_cols:
    transformers.append(("num", SimpleImputer(strategy="median"), numeric_cols))
if categorical_cols:
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", ohe)
    ])
    transformers.append(("cat", cat_pipe, categorical_cols))

preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")

n_splits = min(5, int(y_train.value_counts().min()))
if n_splits < 2:
    raise ValueError("Not enough examples in the smallest training class for CV.")
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
print("Train rows:", len(X_train), "| Test rows:", len(X_test), "| CV folds:", n_splits)


## 5. XGBoost GridSearchCV
The six parameter dimensions below match the cheat sheet exactly. Total combinations: 3 × 3 × 3 × 3 × 2 × 2 = 324. With 5-fold CV, this can require up to 1,620 model fits, so runtime depends on dataset size and hardware.

In [ ]:

param_grid = {
    "model__n_estimators": [50, 100, 200],
    "model__learning_rate": [0.01, 0.1, 0.2],
    "model__max_depth": [3, 5, 7],
    "model__gamma": [0, 0.1, 0.3],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0],
}
print("Expected combinations:", 3 * 3 * 3 * 3 * 2 * 2)

# Label encoding above gives XGBoost consecutive integer class labels.
xgb_params = {
    "random_state": RANDOM_STATE,
    "eval_metric": "logloss",
    "n_jobs": 1,
    "tree_method": "hist",
}
if n_classes > 2:
    xgb_params["objective"] = "multi:softprob"
    xgb_params["num_class"] = n_classes
else:
    xgb_params["objective"] = "binary:logistic"

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", XGBClassifier(**xgb_params))
])

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted",
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring=scoring,
    refit="f1_macro",
    n_jobs=-1,
    return_train_score=True,
    verbose=1,
    error_score="raise"
)

start = time.time()
grid_search.fit(X_train, y_train)
fit_seconds = time.time() - start
best_model = grid_search.best_estimator_

print(f"Grid search completed in {fit_seconds:.1f} seconds.")
print("Best parameters:")
display(pd.Series(grid_search.best_params_, name="best value").to_frame())
print("Best mean CV macro-F1:", round(grid_search.best_score_, 4))


## 6. Inspect top configurations

In [ ]:

cv_results = pd.DataFrame(grid_search.cv_results_).sort_values("rank_test_f1_macro")
show_cols = [
    "rank_test_f1_macro", "mean_test_f1_macro", "std_test_f1_macro",
    "mean_test_accuracy", "mean_test_balanced_accuracy", "mean_fit_time",
    "param_model__n_estimators", "param_model__learning_rate",
    "param_model__max_depth", "param_model__gamma",
    "param_model__subsample", "param_model__colsample_bytree"
]
display(cv_results[show_cols].head(15).reset_index(drop=True))

top = cv_results.head(15).copy()
labels = [
    f"n={r['param_model__n_estimators']}, lr={r['param_model__learning_rate']}, "
    f"d={r['param_model__max_depth']}, g={r['param_model__gamma']}, "
    f"s={r['param_model__subsample']}, c={r['param_model__colsample_bytree']}"
    for _, r in top.iterrows()
]
plt.figure(figsize=(12, 7))
plt.barh(range(len(top))[::-1], top["mean_test_f1_macro"].to_numpy())
plt.yticks(range(len(top))[::-1], labels, fontsize=8)
plt.xlabel("Mean cross-validated macro F1")
plt.title("Top 15 XGBoost configurations")
plt.tight_layout()
plt.show()


## 7. Held-out test evaluation

In [ ]:

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Balanced accuracy": balanced_accuracy_score(y_test, y_pred),
    "Precision (macro)": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "Recall (macro)": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "F1 (macro)": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "F1 (weighted)": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    "MCC": matthews_corrcoef(y_test, y_pred),
    "Cohen's kappa": cohen_kappa_score(y_test, y_pred),
}
try:
    metrics["Log loss"] = log_loss(y_test, y_proba, labels=np.arange(n_classes))
    if n_classes == 2:
        metrics["ROC-AUC"] = roc_auc_score(y_test, y_proba[:, 1])
        metrics["Average precision"] = average_precision_score(y_test, y_proba[:, 1])
    else:
        metrics["ROC-AUC (macro OVR)"] = roc_auc_score(
            y_test, y_proba, multi_class="ovr", average="macro",
            labels=np.arange(n_classes)
        )
except Exception as exc:
    print("Some probability metrics could not be calculated:", exc)

display(pd.Series(metrics, name="test score").to_frame())
print("\nClassification report:")
print(classification_report(
    y_test, y_pred, labels=np.arange(n_classes),
    target_names=class_names, zero_division=0
))

cm = confusion_matrix(y_test, y_pred, labels=np.arange(n_classes))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title("XGBoost — confusion matrix")
plt.tight_layout()
plt.show()


## 8. ROC and precision-recall curves

In [ ]:

if n_classes == 2:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    RocCurveDisplay.from_predictions(y_test, y_proba[:, 1], ax=axes[0])
    axes[0].set_title("ROC curve")
    PrecisionRecallDisplay.from_predictions(y_test, y_proba[:, 1], ax=axes[1])
    axes[1].set_title("Precision-recall curve")
    plt.tight_layout()
    plt.show()
else:
    print("Multiclass classification: macro one-vs-rest ROC-AUC is reported in the metric table where calculable.")


## 9. Feature importance

In [ ]:

try:
    feature_names = best_model.named_steps["preprocess"].get_feature_names_out()
    importances = best_model.named_steps["model"].feature_importances_
    importance_df = pd.DataFrame({
        "feature": feature_names, "importance": importances
    }).sort_values("importance", ascending=False)
    display(importance_df.head(20))
    top_imp = importance_df.head(20).sort_values("importance")
    plt.figure(figsize=(9, 6))
    plt.barh(top_imp["feature"], top_imp["importance"])
    plt.xlabel("Feature importance")
    plt.title("Top 20 XGBoost feature importances")
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print("Could not calculate feature importance:", exc)


## 10. Learning curve
This is an optional, additional diagnostic and may take extra time.

In [ ]:

train_sizes, train_scores, val_scores = learning_curve(
    estimator=best_model,
    X=X_train, y=y_train, cv=cv, scoring="f1_macro",
    train_sizes=np.linspace(0.2, 1.0, 5),
    n_jobs=-1, shuffle=True, random_state=RANDOM_STATE
)
tr_mean, tr_std = train_scores.mean(axis=1), train_scores.std(axis=1)
va_mean, va_std = val_scores.mean(axis=1), val_scores.std(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, tr_mean, marker="o", label="Training macro-F1")
plt.fill_between(train_sizes, tr_mean-tr_std, tr_mean+tr_std, alpha=0.15)
plt.plot(train_sizes, va_mean, marker="o", label="CV macro-F1")
plt.fill_between(train_sizes, va_mean-va_std, va_mean+va_std, alpha=0.15)
plt.xlabel("Number of training examples")
plt.ylabel("Macro F1")
plt.title("XGBoost learning curve")
plt.legend()
plt.tight_layout()
plt.show()


## 11. Optional paired significance check: tuned vs default XGBoost
This is only a rough diagnostic: the number of folds is small, and the tuned model was selected using the same training data, so the comparison is subject to model-selection optimism.

In [ ]:

default_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", XGBClassifier(**xgb_params))
])
tuned_scores = cross_validate(
    best_model, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1
)["test_score"]
default_scores = cross_validate(
    default_model, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1
)["test_score"]

print("Tuned fold macro-F1:", np.round(tuned_scores, 4))
print("Default fold macro-F1:", np.round(default_scores, 4))
print("Mean tuned:", round(float(tuned_scores.mean()), 4))
print("Mean default:", round(float(default_scores.mean()), 4))
try:
    t_stat, t_p = ttest_rel(tuned_scores, default_scores)
    print(f"Paired t-test: statistic={t_stat:.4f}, p-value={t_p:.4f}")
except Exception as exc:
    print("Paired t-test unavailable:", exc)
try:
    if np.allclose(tuned_scores, default_scores):
        print("Wilcoxon skipped because paired scores are identical.")
    else:
        w_stat, w_p = wilcoxon(tuned_scores, default_scores)
        print(f"Wilcoxon signed-rank: statistic={w_stat:.4f}, p-value={w_p:.4f}")
except Exception as exc:
    print("Wilcoxon unavailable:", exc)


## 12. Export CSV results

In [ ]:

cv_results.to_csv("xgboost_grid_search_results.csv", index=False)

prediction_df = pd.DataFrame({
    "actual_encoded": y_test.to_numpy(),
    "predicted_encoded": np.asarray(y_pred, dtype=int),
    "actual_label": label_encoder.inverse_transform(y_test.to_numpy()),
    "predicted_label": label_encoder.inverse_transform(np.asarray(y_pred, dtype=int))
}, index=X_test.index)
prediction_df.to_csv("xgboost_test_predictions.csv", index=True)

summary = pd.DataFrame([{
    "model": "XGBClassifier",
    "best_cv_macro_f1": grid_search.best_score_,
    "test_accuracy": metrics.get("Accuracy", np.nan),
    "test_balanced_accuracy": metrics.get("Balanced accuracy", np.nan),
    "test_macro_f1": metrics.get("F1 (macro)", np.nan),
    "fit_seconds": fit_seconds,
    "best_params": str(grid_search.best_params_)
}])
summary.to_csv("xgboost_summary.csv", index=False)

print("Saved xgboost_grid_search_results.csv")
print("Saved xgboost_test_predictions.csv")
print("Saved xgboost_summary.csv")
display(summary)


## Quick viva notes
- **XGBoost** is an optimized gradient-boosting implementation with regularization and efficient tree-building options.
- **`n_estimators`**: number of boosting rounds/trees.
- **`learning_rate`**: scales each boosting step.
- **`max_depth`**: maximum depth of each tree; higher values can increase complexity and overfitting.
- **`gamma`**: minimum loss reduction required to make a further split; larger values make splitting more conservative.
- **`subsample`**: fraction of training rows sampled for each boosting round.
- **`colsample_bytree`**: fraction of features sampled per tree.
- **`eval_metric="logloss"`**: evaluation metric configured for XGBoost training.
- **Why use a pipeline?** It prevents preprocessing from leaking information across CV folds.
- **Why keep a test set?** It estimates generalization after hyperparameter selection.
- **What may need changing in the exam?** CSV file path and exact target column. Keep the cheat-sheet grid unchanged unless the question explicitly asks for a different grid.